# Exercise: Data Preparation for nanoGPT

Welcome! In this exercise, you'll get hands-on experience with the first critical step of training a language model: preparing the data. You'll build a simple character-level tokenizer and use it to convert a text corpus into a format that's ready for training. This process is fundamental to how models like GPT understand and process language.

In this exercise you will:
*   Implement a simple character-level tokenizer from scratch.
*   Use your tokenizer to convert raw text into a sequence of integer token IDs.
*   Save the tokenized data to a binary file for efficient loading during training.

Let's get started!

In [ ]:
import os
import numpy as np
import pickle

# --- Setup: Helper code and data ---
# We'll create a dummy data directory and a small text file for you to work with.
# In a real project, this would be a much larger corpus, like Shakespeare or Wikipedia.

DATA_DIR = "data"
INPUT_FILE = os.path.join(DATA_DIR, "input.txt")
TOKENIZER_FILE = os.path.join(DATA_DIR, "tokenizer.pkl")
TRAIN_FILE = os.path.join(DATA_DIR, "train.bin")
VAL_FILE = os.path.join(DATA_DIR, "val.bin")


# Create the data directory if it doesn't exist
os.makedirs(DATA_DIR, exist_ok=True)

# Create a sample text file
sample_text = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?
"""
with open(INPUT_FILE, "w") as f:
    f.write(sample_text)

print(f"Created sample text file at: {INPUT_FILE}")
print("------")
print("File content:")
print(sample_text)
print("------")

# You can now use the `sample_text` variable or load the file from `INPUT_FILE`.

### Exercise 1: Implement a Character-Level Tokenizer

Your first task is to create a simple tokenizer. This tokenizer will scan a body of text, find all the unique characters, and create a mapping from each character to a unique integer ID. This is the foundation of turning text into numbers that a model can process.

You will need to implement two key components:
1.  `stoi` (string-to-integer): A dictionary mapping each character to its integer ID.
2.  `itos` (integer-to-string): A list or dictionary mapping each integer ID back to its character.

**Hint:** To get all unique characters from a text string, you can use `sorted(list(set(text)))`. This gives you a consistent, sorted vocabulary.

In [ ]:
class SimpleCharTokenizer:
    """A simple character-level tokenizer."""
    def __init__(self, text: str):
        """
        Initializes the tokenizer and builds the vocabulary.
        
        Args:
            text (str): The text corpus to build the vocabulary from.
        """
        self.stoi = {}
        self.itos = {}
        self.chars = []
        
        ### START CODE HERE ### (≈ 4 lines)
        self.chars = sorted(list(set(text)))
        self.stoi = { ch:i for i,ch in enumerate(self.chars) }
        self.itos = { i:ch for i,ch in enumerate(self.chars) }
        ### END CODE HERE ###
    
    @property
    def vocab_size(self) -> int:
        """Returns the size of the vocabulary."""
        return len(self.chars)

    def encode(self, s: str) -> list[int]:
        """
        Converts a string to a list of token IDs.
        
        Args:
            s (str): The input string.
            
        Returns:
            list[int]: A list of integer token IDs.
        """
        ### START CODE HERE ### (≈ 1 line)
        return [self.stoi[c] for c in s]
        ### END CODE HERE ###

    def decode(self, ids: list[int]) -> str:
        """
        Converts a list of token IDs back to a string.
        
        Args:
            ids (list[int]): A list of integer token IDs.
            
        Returns:
            str: The decoded string.
        """
        ### START CODE HERE ### (≈ 1 line)
        return ''.join([self.itos[i] for i in ids])
        ### END CODE HERE ###

In [ ]:
# --- Test Cell for Exercise 1 ---
print("Running tests for SimpleCharTokenizer...")

# 1. Test with a simple string
test_text = "hello world"
tokenizer = SimpleCharTokenizer(test_text)

# Test vocabulary size
expected_vocab_size = 10 # h, e, l, o,  , w, r, d
assert tokenizer.vocab_size == expected_vocab_size, \
    f"FAIL: vocab_size. Got {tokenizer.vocab_size}, expected {expected_vocab_size}"

# Test encoding
encoded_text = tokenizer.encode("hello")
# Expected encoding depends on the sorted order of chars: ' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w'
# h=3, e=2, l=4, l=4, o=5
expected_encoded = [3, 2, 4, 4, 5]
assert encoded_text == expected_encoded, \
    f"FAIL: encode. Got {encoded_text}, expected {expected_encoded}"

# Test decoding
decoded_text = tokenizer.decode(expected_encoded)
expected_decoded = "hello"
assert decoded_text == expected_decoded, \
    f"FAIL: decode. Got '{decoded_text}', expected '{expected_decoded}'"

# 2. Test round-trip consistency with the larger sample text
full_tokenizer = SimpleCharTokenizer(sample_text)
round_trip_text = full_tokenizer.decode(full_tokenizer.encode(sample_text))
assert sample_text == round_trip_text, \
    "FAIL: Round-trip test. Decoding the encoded text did not yield the original text."

print("✅ All tests passed!")

### Exercise 2: Prepare and Save Tokenized Data

Great job! Now that you have a working tokenizer, the next step is to process your entire dataset and save it in an efficient format.

Your task is to implement the `prepare_and_save_data` function. This function will:
1.  Take the raw text and your tokenizer as input.
2.  Encode the entire text into a sequence of token IDs.
3.  Convert this sequence into a NumPy array. For memory efficiency, we'll use the `np.uint16` data type, which is suitable for vocabularies up to 65,535 tokens.
4.  Save this NumPy array to a binary file at the specified output path.

In [ ]:
def prepare_and_save_data(text: str, tokenizer: SimpleCharTokenizer, output_path: str) -> None:
    """
    Tokenizes text and saves the resulting token IDs to a binary file.
    
    Args:
        text (str): The raw text to process.
        tokenizer (SimpleCharTokenizer): The tokenizer to use.
        output_path (str): The path to save the binary file.
    """
    ### START CODE HERE ### (≈ 3 lines)
    # 1. Encode the text
    token_ids = tokenizer.encode(text)
    
    # 2. Convert to a NumPy array with the specified dtype
    token_ids_array = np.array(token_ids, dtype=np.uint16)
    
    # 3. Save the array to a binary file
    token_ids_array.tofile(output_path)
    ### END CODE HERE ###
    print(f"Saved {len(token_ids_array)} tokens to {output_path}")

In [ ]:
# --- Test Cell for Exercise 2 ---
print("Running tests for prepare_and_save_data...")

# Setup
test_text = "test data"
test_tokenizer = SimpleCharTokenizer(test_text)
test_output_path = os.path.join(DATA_DIR, "test.bin")

# Run the function
prepare_and_save_data(test_text, test_tokenizer, test_output_path)

# 1. Check if the file was created
assert os.path.exists(test_output_path), f"FAIL: File '{test_output_path}' was not created."

# 2. Load the file and check its contents
loaded_ids = np.fromfile(test_output_path, dtype=np.uint16)
expected_ids = np.array(test_tokenizer.encode(test_text), dtype=np.uint16)

assert np.array_equal(loaded_ids, expected_ids), \
    f"FAIL: Content mismatch. Loaded {loaded_ids}, expected {expected_ids}"

# 3. Check the data type
assert loaded_ids.dtype == np.uint16, f"FAIL: Dtype mismatch. Got {loaded_ids.dtype}, expected {np.uint16}"

# Clean up the test file
os.remove(test_output_path)
print("✅ All tests passed!")

### Challenge (Optional): Splitting the Data

In machine learning, it's crucial to hold out a portion of your data for validation. This helps you check if your model is truly learning or just memorizing the training data.

For this optional challenge, implement the `split_data` function. It should take a NumPy array of token IDs and split it into a training set and a validation set based on a given ratio. A 90/10 split is a common practice.

In [ ]:
def split_data(token_ids: np.ndarray, train_ratio: float = 0.9) -> tuple[np.ndarray, np.ndarray]:
    """
    Splits a NumPy array of token IDs into training and validation sets.
    
    Args:
        token_ids (np.ndarray): The full array of token IDs.
        train_ratio (float): The proportion of data to use for training (e.g., 0.9 for 90%).
        
    Returns:
        tuple[np.ndarray, np.ndarray]: A tuple containing the training data and validation data.
    """
    ### START CODE HERE ### (≈ 3 lines)
    n = len(token_ids)
    split_idx = int(n * train_ratio)
    train_data = token_ids[:split_idx]
    val_data = token_ids[split_idx:]
    return train_data, val_data
    ### END CODE HERE ###

# --- Test Cell for Challenge ---
print("Running tests for split_data...")

# 1. Test with a simple array
data = np.arange(100) # An array from 0 to 99
train_ids, val_ids = split_data(data, train_ratio=0.9)

assert len(train_ids) == 90, f"FAIL: Incorrect train split size. Got {len(train_ids)}, expected 90."
assert len(val_ids) == 10, f"FAIL: Incorrect validation split size. Got {len(val_ids)}, expected 10."
assert train_ids[-1] == 89, f"FAIL: Incorrect train data content. Last element was {train_ids[-1]}, expected 89."
assert val_ids[0] == 90, f"FAIL: Incorrect validation data content. First element was {val_ids[0]}, expected 90."

# 2. Test with an odd-sized array
data_odd = np.arange(101)
train_ids_odd, val_ids_odd = split_data(data_odd, train_ratio=0.9)
# 101 * 0.9 = 90.9, int(90.9) = 90
assert len(train_ids_odd) == 90, f"FAIL: Incorrect train split size for odd array. Got {len(train_ids_odd)}, expected 90."
assert len(val_ids_odd) == 11, f"FAIL: Incorrect validation split size for odd array. Got {len(val_ids_odd)}, expected 11."

print("✅ All challenge tests passed!")

In [ ]:
# --- Integration: Putting It All Together ---
# This cell uses all the functions you've built to perform the full data preparation pipeline.
# No need to edit anything here, just run it to see your work in action!

print("--- Starting Full Data Preparation Pipeline ---")

# 1. Load the raw text data
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    text = f.read()
print(f"Loaded {len(text)} characters from {INPUT_FILE}")

# 2. Create and save the tokenizer based on our text
tokenizer = SimpleCharTokenizer(text)
with open(TOKENIZER_FILE, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Saved tokenizer to {TOKENIZER_FILE}")

# 3. Tokenize the entire dataset
all_token_ids = np.array(tokenizer.encode(text), dtype=np.uint16)

# 4. Split the data into training and validation sets (using your optional function)
try:
    train_ids, val_ids = split_data(all_token_ids)
    
    # 5. Save the training and validation data to binary files
    train_ids.tofile(TRAIN_FILE)
    val_ids.tofile(VAL_FILE)
    
    print(f"Split data: {len(train_ids)} training tokens, {len(val_ids)} validation tokens.")
    print(f"Saved training data to {TRAIN_FILE}")
    print(f"Saved validation data to {VAL_FILE}")

except NameError:
    print("\nSkipping data split and save because `split_data` is not defined.")
    print("Complete the optional challenge to run this part!")


print("\n--- Pipeline Finished ---")
print("Congratulations! You have successfully prepared the data for training a nanoGPT-style model.")
print("The created files in the 'data/' directory are now ready to be consumed by a training script.")